# Drive a screw-shaped magnetic robot through a tube

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/v0.3-colab-notebooks/docs/notebooks/tutorial_a.ipynb)

The robot is a small plastic screw, 7.5 millimetres long and 3 to 4 millimetres across, with two tiny magnets inside.
It comes from a real experiment: de Jongh et al. (2025), ["Swimming dynamics of screw-shaped untethered magnetic
robots in confined spaces"](https://doi.org/10.1007/s11071-025-11646-7), *Nonlinear Dynamics*. It lies on the floor of a tube of water a quarter of an
inch (6.35 mm) wide.

How it swims: a magnetic field turns in a circle around the tube. The magnets inside the robot try to line up with the
field, the way a compass needle follows a magnet, so the robot turns with it. As the screw turns, its thread pushes
against the water like a corkscrew in a cork, and the robot moves along the tube. The 3D view will show the robot
turning and moving along the tube.

Run the cells one by one from the top (click a cell, then press Shift+Enter). Nothing needs installing on your
computer.

**Install.** This installs MIME. Allow a minute.

In [ ]:
!pip install -q --pre "mime-engine==0.3.0.dev1"

**Load the pieces we use.** Every number in MIME carries its unit, so the units are imported too: `mm` is
millimetres, `Hz` turns per second, `mT` millitesla (the strength of a magnetic field).

In [ ]:
import numpy as np

from mime import Apparatus, Body, Experiment, Medium, State, Structure, viz
from mime.data import asset
from mime.dynamics import RigidInertial
from mime.fields import UniformRotating
from mime.interactions import KEPT, Contact, Drag, Gravity, Magnetic
from mime.observations import SwimSpeed
from mime.scene import Scene, Screw, Tube, X, Z
from mime.things import Part
from mime.units import (A_m2, Hz, dimensionless, kg, kg_m2, kg_per_m3, m_per_s, m_per_s2, mm, mm_per_s, mPa_s, ms,
                        mT, rad_per_s, s)

**The settings.** The numbers you will change later, in "Try it yourself". The field is about twenty times as strong
as Earth's magnetic field.

In [ ]:
field_turns_per_second = 10.0  # how fast the magnetic field turns around the tube
field_strength = 1.2           # millitesla
run_time = 5.0                 # seconds

**Start the experiment.** An experiment holds the whole world. Here "up" is the z direction, gravity is Earth's, and
MIME moves time forward in small steps of half a millisecond.

In [ ]:
experiment = Experiment("screw_robot", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(9.81), dt=ms(0.5))

**The tube and the water.** A tube 100 mm long lying along the x direction, filled with water. We tell MIME the
water's density and its viscosity, which is how thick a liquid is (honey's is high, water's low). MIME asks where each
number comes from, so we say.

In [ ]:
tube = experiment.add_structure(Structure(
    "tube", geometry=Tube(axis=+X, inner_radius=mm(3.175), wall=mm(1.6), length=mm(100.0))))
water = experiment.add_medium(Medium("water", fills=tube.lumen))
experiment.declare("water.density", kg_per_m3(1000.0), reason="water")
experiment.declare("water.viscosity", mPa_s(1.0), reason="water")

**The robot.** Its shape is the screw from the paper (its radius, its length, and the numbers that shape its thread,
from the paper's Table 1). It moves as one solid piece. It starts on the floor of the tube, its centre 1 mm below the
tube's centre line, lying along the tube and not moving. It is a little denser than water (resin with two magnets
inside), and its magnets point across it, along y. We give it the same resistance to turning in every direction, as
the original model did, so that the quick wall rule below keeps it upright: with its real, uneven shape, that rule
would let it tip across the tube, which it cannot do in a real tube.

In [ ]:
fl9 = Screw(radius=mm(1.56), length=mm(7.47), wavenumber=dimensionless(2.33), modulation=dimensionless(0.33),
            starts=2)
robot = experiment.add_body(Body(
    "robot",
    dynamics=RigidInertial(),
    initial=State(position=mm((0.0, 0.0, -1.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, 0.0)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
    parts={"screw": Part(fl9)},
))
experiment.declare("robot.parts.screw.density", kg_per_m3(1410.0), reason="resin with two magnets inside")
experiment.declare("robot.mass", kg(8.49e-5), reason="the original model's")
experiment.declare("robot.inertia", kg_m2((1.03e-10, 1.03e-10, 1.03e-10)),
                   reason="the same in every direction, as the original model's")
experiment.declare("robot.magnetic_moment", A_m2((0.0, 1.68e-3, 0.0)), reason="two magnets, across the robot")

**The turning magnetic field, and its twist on the robot.** The field is the same everywhere in the tube and turns
around the tube's axis (real setups make one with coils of wire around the tube). The robot's magnets try to line up
with it, so the field twists the robot, and the robot turns with the field.

In [ ]:
field = experiment.add_apparatus(Apparatus(
    "field",
    fields={"magnetic": UniformRotating(rotation_axis=tube.axis, initial_direction=(0.0, 1.0, 0.0))},
    commands={"frequency": Hz(field_turns_per_second), "amplitude": mT(field_strength)},
))
experiment.attach(Magnetic.Torque(), body=robot, field=field.magnetic)

**The water's push.** Water pushes back on anything moving through it (this push is called drag). Because the
screw's thread is slanted, turning the screw also pushes it along the tube. The model of the water's push used here is
a fast learned approximation, a small neural network that ships with MIME. It is known to be off near the tube's wall,
where this robot swims, so the speed you will see is not yet accurate. A corrected one comes in the next development
release, 0.3.0.dev2.

In [ ]:
experiment.attach(Drag.Surrogate(weights=asset("dejongh_mlp_v2")), body=robot, medium=water)

# A slower, more faithful model of the water's push, the MRS model, works the push out from the flow itself. Swapping
# it in changes this one line and nothing else: that is MIME's modularity. It needs the tube's data file first
# (!mime assets fetch unit_tube_R0.997, 209 MB), and it is very slow: one step took about half a minute on a laptop's
# processor, so the 5 second run would take days on Colab's free computers. It is made to run on a GPU, where it
# becomes practical: on a GPU runtime (Colab's T4, when one is free) it runs many times faster.
# To swap the model, put a # before the first line of this cell, and uncomment the next line:

# experiment.attach(Drag.MRS(wall_kind="tube"), body=robot, medium=water)

**Gravity, and the wall holding the robot in.** Gravity pulls the robot onto the floor of the tube, and the wall
holds it there, 0.1 mm off the surface. The wall rule here is a simple one MIME keeps from its earlier versions: it
keeps the robot's centre far enough from the wall, which is fast, and enough while the robot lies along the tube.

In [ ]:
experiment.attach(Gravity(), body=robot, medium=water)
SimpleWall, _ = KEPT["contact", "0.1.0"]
experiment.attach(SimpleWall.Projection(margin=mm(0.1)), body=robot, surface=tube.inner_surface)

# The full wall rule follows the screw's whole shape, not just its centre: it is more faithful, and slower. The
# 5 second run takes about four minutes on Colab's free computers instead of a few seconds. To use it, put a # before
# the quick wall rule's line above, and uncomment the next line:

# experiment.attach(Contact.Projection(margin=mm(0.1)), body=robot, surface=tube.inner_surface)

**Run it.** First we tell MIME what to measure: the robot's speed along the tube. Then MIME checks the experiment and
builds it, and runs 5 seconds of the robot's life in 10,000 small steps, keeping where the robot is at every step
(`history=True`) and a 3D frame every 10 milliseconds for the view. Its display shows it compiling (turning the
experiment into fast code), then stepping through time, then saving the run to a folder, `robot_run`. This takes
under a minute on Colab.

When it builds, MIME prints a warning: the robot turns faster than the range its model of the water's push was made
for. That is the same limit as above.

In [ ]:
swimming = experiment.observe(SwimSpeed(robot, along=tube.axis))
built = experiment.build()
run = built.run_to("robot_run", s(run_time), record=[swimming], history=True, stage_every=ms(10.0))
speed = run.result[swimming].speed.between(s(run_time / 2)).mean().to(mm_per_s)
print(f"Over the second half of the run, the robot swam at {speed:.2f} millimetres per second.")

**Plot the swim.** The robot's position along the tube at each moment. MIME kept where the robot was at every step
(its `history`), in metres. Matplotlib, which Colab has, draws the plot.

In [ ]:
import matplotlib.pyplot as plt

time = run.result.time.to(s)
along = np.asarray(run.result.history["robot"]["pose"])[:, 0] * 1000  # from metres to millimetres

plt.figure(figsize=(8, 4))
plt.plot(time, along, color="#2a78d6", linewidth=2)
plt.xlabel("time (s)")
plt.ylabel("position along the tube (mm)")
plt.title("The robot swimming along the tube")
plt.grid(color="#e1e0d9")
plt.show()

**Watch it in 3D.** Drag to turn the view, scroll to zoom, and use the play button or the slider to move through time.
The view needs an internet connection.

In [ ]:
viz.display("robot_run")

## Try it yourself

Change a number in the settings cell, then run every cell again (Runtime, then Run all).

- **Turn the field faster:** `field_turns_per_second = 20.0`. The robot turns twice as fast and swims about twice as
  far: about 33 mm in 5 seconds instead of about 16.
- **Turn it much faster:** `field_turns_per_second = 50.0`. Now the robot cannot keep up: it slips behind the field,
  turns only about once a second, and moves about 3 mm in 5 seconds.
- **Turn it the other way:** `field_turns_per_second = -10.0`. The robot turns the other way and swims backwards,
  about 16 mm in 5 seconds.

The fastest the field can turn with the robot still following is called the **step-out frequency**, one of the most
important topics in helical microrobotics. Below it, the faster the field turns, the faster the robot swims. Beyond
it, the magnet's twist can no longer beat the water's resistance: the robot slips behind the field and swims slower,
not faster. In this simulation it lies between 42 and 45 turns a second: at 42 the robot still follows and swims about
15 mm a second, and at 45 it slips. Step-out is not programmed into MIME: it comes out of the physics you added, the
magnet's twist, the water's push and the robot's inertia.

## Learn more

- The robot and the experiments it comes from: Luuc de Jongh, Anke Klingner, Leendert-Jan W. Ligtenberg, Marcus C. J.
  de Boer, Jaap van der Kooij, Roger Lomme, Dorothee Wasserberg, H. Remco Liefers, Pascal Jonkheijm, Michiel C. Warlé
  and Islam S. M. Khalil, "Swimming dynamics of screw-shaped untethered magnetic robots in confined spaces",
  *Nonlinear Dynamics* 113, 29197-29213 (2025), [doi:10.1007/s11071-025-11646-7](https://doi.org/10.1007/s11071-025-11646-7) (open access, CC BY 4.0).
- [MIME's documentation](https://microrobotica.org/mime/0.3.0/): the user guide, its tutorials, and how to read what a run gives back.
- [The example this notebook comes from](https://github.com/Microrobotics-Simulation-Framework/MIME/blob/v0.3.0.dev1/examples/script_a.py), with every number's source.
- The first notebook: [watch a steel ball settle in a tube of water](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/v0.3-colab-notebooks/docs/notebooks/sphere_settling_tube.ipynb).